In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 뉴스 요약 분석 시스템 - LEAN VERSION (키워드 리스트 저장)
# EXAONE 4.0.1+GGUF+Q4KL Non-reasoning 모드 + Greedy Decoding
# ═══════════════════════════════════════════════════════════════════════
import os
import pandas as pd
import openai
from tqdm import tqdm
import re
import json
import time
import logging

# ═══════════════════════════════════════════════════════════════════════
# 1. 로깅 설정 (상세 모드)
# ═══════════════════════════════════════════════════════════════════════
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("news_summary_analysis.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

# ═══════════════════════════════════════════════════════════════════════
# 2. 프롬프트 (EXAONE 4.0.1 최적화)
# ═══════════════════════════════════════════════════════════════════════
SUMMARY_PROMPT = """다음 뉴스 기사를 3줄 이하로 요약하세요.

**[작업]**
1. 뉴스 기사를 3줄 이내로 요약
2. 핵심 키워드를 정확히 3개 추출
3. 중립적인 톤과 문어체 사용

**[출력 형식]**
{{
  "summary": "요약문",
  "keywords": ["키워드1", "키워드2", "키워드3"]
}}

**[뉴스 기사]**
{news_text}

JSON만 출력하세요. <think> 태그 사용 금지."""

# ═══════════════════════════════════════════════════════════════════════
# 3. 유틸리티 함수 (핵심만)
# ═══════════════════════════════════════════════════════════════════════
def clean_text(text):
    """텍스트 정제 (제어문자 완전 제거)"""
    if pd.isna(text) or text == "":
        return "본문 정보 없음"

    text = str(text)
    # 제어문자 제거
    text = text.replace('\n', ' ')
    text = text.replace('\r', ' ')
    text = text.replace('\t', ' ')
    text = text.replace('\b', ' ')
    text = text.replace('\f', ' ')
    text = text.replace('\v', ' ')
    # 특수 시퀀스 제거
    text = re.sub(r'/[a-z]', '', text)
    # 모든 제어문자 제거
    text = re.sub(r'[\x00-\x1f\x7f-\x9f]', ' ', text)
    # 연속 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else "본문 정보 없음"

def parse_json(response):
    """JSON 파싱 (간소화)"""
    try:
        # 제어문자 제거
        response = clean_text(response)
        # Markdown 코드 블록 제거
        response = re.sub(r'```json\s*', '', response)
        response = re.sub(r'```\s*', '', response)
        # JSON 추출
        match = re.search(r'\{.*\}', response, re.DOTALL)
        if match:
            result = json.loads(match.group(0))

            # 필드 검증
            summary = result.get("summary", "").strip()
            keywords = result.get("keywords", [])

            # 키워드 정리
            if not isinstance(keywords, list):
                if isinstance(keywords, str):
                    keywords = [k.strip() for k in keywords.split(',')][:3]
                else:
                    keywords = []

            # 키워드 개수 조정 (정확히 3개)
            keywords = keywords[:3]
            while len(keywords) < 3:
                keywords.append("")

            return summary, keywords

        return "JSON 형식 오류", ["", "", ""]

    except json.JSONDecodeError as e:
        logger.warning(f"JSON 파싱 실패: {e}")
        return f"JSON 파싱 실패: {str(e)}", ["", "", ""]
    except Exception as e:
        logger.error(f"예외 발생: {e}")
        return f"오류: {str(e)}", ["", "", ""]

# ═══════════════════════════════════════════════════════════════════════
# 4. 뉴스 요약 분석기 (EXAONE 4.0.1)
# ═══════════════════════════════════════════════════════════════════════

class NewsSummaryAnalyzer:
    """EXAONE 4.0.1 기반 뉴스 요약 분석기 (Non-reasoning + Greedy)"""

    def __init__(self, client, output_dir='./'):
        self.client = client
        self.output_dir = output_dir
        os.makedirs(output_dir, exist_ok=True)

        logger.info("🚀 EXAONE 4.0.1 뉴스 요약 분석기 초기화 완료!")
        logger.info("⚙️  설정: Non-reasoning 모드 + Greedy Decoding (temp=0.0)")
        logger.info("🔑 키워드 형식: 단일 컬럼 (쉼표 구분 문자열)")

    def analyze_news(self, news_text):
        """
        단일 뉴스 요약 및 키워드 추출

        Args:
            news_text: 뉴스 본문

        Returns:
            tuple: (summary, keywords)
        """
        prompt = SUMMARY_PROMPT.format(news_text=news_text)

        try:
            response = self.client.chat.completions.create(
                model="lgai-exaone_exaone-4.0.1-32b",  # EXAONE 4.0.1
                messages=[
                    {
                        "role": "system",
                        "content": "당신은 뉴스 요약 전문가입니다. JSON만 출력하세요."
                    },
                    {"role": "user", "content": prompt}
                ],
                # ═══════════════════════════════════════════════════
                # EXAONE 4.0.1 Non-reasoning + Greedy Decoding
                # ═══════════════════════════════════════════════════
                temperature=0.0,        # Greedy decoding
                max_tokens=8196,

                # Non-reasoning 모드 활성화
                extra_body={
                    "chat_template_kwargs": {
                        "enable_thinking": False  # 핵심!
                    }
                }
            )

            model_response = response.choices[0].message.content.strip()

            # 디버깅 로그
            logger.debug(f"EXAONE 4.0.1 응답: {model_response[:200]}")

            # JSON 파싱
            summary, keywords = parse_json(model_response)

            # 결과 로그
            logger.info(f"✅ 요약 완료: {summary[:50]}...")
            logger.info(f"🎯 키워드: {keywords}")

            return summary, keywords

        except Exception as e:
            logger.error(f"❌ EXAONE 4.0 분석 오류: {e}")
            return f"분석 오류: {str(e)}", ["", "", ""]

    def process_dataframe(self, df, output_file, checkpoint_interval=50):
        """
        데이터프레임 배치 처리

        Args:
            df: 입력 데이터프레임
            output_file: 출력 파일명
            checkpoint_interval: 체크포인트 저장 간격

        Returns:
            DataFrame: 분석 완료된 데이터프레임
        """
        output_path = os.path.join(self.output_dir, output_file)
        checkpoint_path = os.path.join(self.output_dir, f"checkpoint_{output_file}")

        # 결과 저장용
        summaries = []
        keywords_list = []

        # 체크포인트 복구
        start_idx = 0
        if os.path.exists(checkpoint_path):
            try:
                cp_df = pd.read_csv(checkpoint_path, encoding='utf-8-sig')

                # 새로운 형식(keywords 단일 컬럼) 또는 기존 형식(keyword1/2/3) 모두 지원
                has_new_format = 'keywords' in cp_df.columns
                has_old_format = 'keyword1' in cp_df.columns

                if 'summary' in cp_df.columns and (has_new_format or has_old_format):
                    completed = sum(~cp_df['summary'].isna())
                    if completed > 0:
                        start_idx = completed

                        # 기존 결과 복구
                        for i in range(start_idx):
                            summaries.append(cp_df['summary'].iloc[i])

                            # 새로운 형식: keywords 컬럼
                            if has_new_format:
                                kw_str = cp_df['keywords'].iloc[i]
                                if pd.notna(kw_str) and kw_str:
                                    # 쉼표로 구분된 문자열을 리스트로 변환
                                    kw_list = [k.strip() for k in str(kw_str).split(',')]
                                    # 정확히 3개 유지
                                    while len(kw_list) < 3:
                                        kw_list.append("")
                                    keywords_list.append(kw_list[:3])
                                else:
                                    keywords_list.append(["", "", ""])

                            # 기존 형식: keyword1/2/3 컬럼
                            elif has_old_format:
                                keywords_list.append([
                                    cp_df['keyword1'].iloc[i] if pd.notna(cp_df['keyword1'].iloc[i]) else "",
                                    cp_df['keyword2'].iloc[i] if pd.notna(cp_df['keyword2'].iloc[i]) else "",
                                    cp_df['keyword3'].iloc[i] if pd.notna(cp_df['keyword3'].iloc[i]) else ""
                                ])

                        logger.info(f"📂 체크포인트 복구: {start_idx}행부터 재개")
            except Exception as e:
                logger.warning(f"⚠️  체크포인트 복구 실패: {e}")

        # 통계 변수
        success_count = len(summaries)
        failure_count = 0

        # 진행 상황 표시
        pbar = tqdm(
            total=len(df) - start_idx,
            initial=0,
            desc="🚀 EXAONE 4.0.1 뉴스 요약"
        )

        # 데이터 순차 처리
        for i in range(start_idx, len(df)):
            news_text = df.iloc[i]['text']

            # 텍스트 정제 및 검증
            if pd.isna(news_text) or news_text == "" or news_text == "본문 정보 없음":
                news_text = "본문 정보 없음"

            logger.info(f"{'='*60}")
            logger.info(f"📰 기사 {i+1}/{len(df)} 처리 중")
            logger.info(f"{'='*60}")

            # 요약 분석 (재시도 1회)
            try:
                summary, keywords = self.analyze_news(news_text)

                # 실패 감지
                if any(word in summary for word in ["실패", "오류", "파싱 실패"]):
                    logger.warning(f"⚠️  1차 시도 실패, 재시도 중...")
                    time.sleep(1)
                    summary, keywords = self.analyze_news(news_text)

            except Exception as e:
                logger.error(f"❌ 행 {i} 처리 중 예외: {e}")
                summary = f"처리 오류: {str(e)}"
                keywords = ["", "", ""]

            # 결과 검증 및 카운팅
            if any(word in summary for word in ["실패", "오류", "파싱 실패"]):
                failure_count += 1
                logger.warning(f"⚠️  처리 실패! (누적 실패: {failure_count}개)")
            else:
                success_count += 1
                logger.info(f"✅ 처리 성공! (누적 성공: {success_count}개)")

            # 결과 저장
            summaries.append(summary)
            keywords_list.append(keywords)

            # 진행 상황 로그
            logger.info(f"📊 현재 진행률: {i+1}/{len(df)} (성공: {success_count}, 실패: {failure_count})")

            # 진행 바 업데이트
            pbar.update(1)

            # 체크포인트 저장 (새로운 형식: keywords 단일 컬럼)
            if (i + 1) % checkpoint_interval == 0 or i == len(df) - 1:
                temp_df = df.copy()
                temp_df['summary'] = summaries + [''] * (len(temp_df) - len(summaries))

                # 키워드를 쉼표로 구분된 단일 문자열로 저장
                temp_df['keywords'] = [', '.join(kw) for kw in keywords_list] + [''] * (len(temp_df) - len(keywords_list))

                temp_df.to_csv(checkpoint_path, encoding='utf-8-sig', index=False)
                logger.info(f"💾 체크포인트 저장: {len(summaries)}/{len(df)}")

        pbar.close()

        # 최종 결과 생성 (새로운 형식: keywords 단일 컬럼)
        result_df = df.copy()
        result_df['summary'] = summaries

        # 키워드를 쉼표로 구분된 단일 문자열로 저장
        result_df['keywords'] = [', '.join(kw) for kw in keywords_list]

        # 최종 통계
        final_success_rate = (success_count / len(df)) * 100

        logger.info(f"\n{'='*60}")
        logger.info("🎉 EXAONE 4.0.1 뉴스 요약 분석 완료!")
        logger.info(f"{'='*60}")
        logger.info(f"📊 총 분석: {len(df)}개")
        logger.info(f"✅ 성공: {success_count}개")
        logger.info(f"❌ 실패: {failure_count}개")
        logger.info(f"📈 성공률: {final_success_rate:.2f}%")
        logger.info(f"💾 결과 저장: {output_path}")
        logger.info(f"🔑 키워드 형식: 단일 컬럼 'keywords' (쉼표 구분)")
        logger.info(f"{'='*60}\n")

        # 최종 저장
        result_df.to_csv(output_path, encoding='utf-8-sig', index=False)

        return result_df

# ═══════════════════════════════════════════════════════════════════════
# 5. 데이터 로드 함수
# ═══════════════════════════════════════════════════════════════════════

def load_and_clean_data(file_path):
    """데이터 로드 및 정제"""
    logger.info(f"📚 데이터 로드 중: {file_path}")
    df = pd.read_csv(file_path, encoding="utf-8-sig")

    logger.info("="*60)
    logger.info("📋 원본 데이터 정보")
    logger.info("="*60)
    logger.info(f"총 행 수: {len(df)}")
    logger.info(f"text 컬럼 NaN: {df['text'].isna().sum()}개")
    logger.info(f"text 컬럼 빈 문자열: {(df['text'] == '').sum()}개")

    # 텍스트 정제
    df['text'] = df['text'].apply(clean_text)

    logger.info(f"✅ 데이터 정제 완료: {len(df)}개 기사")
    logger.info(f"⚠️  '본문 정보 없음' 처리: {(df['text'] == '본문 정보 없음').sum()}개")

    # 샘플 확인
    logger.info("\n📰 정제된 데이터 샘플:")
    for i in range(min(3, len(df))):
        text = df.iloc[i]['text']
        logger.info(f"기사 {i+1}: {str(text)[:80]}...")

    logger.info("="*60 + "\n")

    return df

# ═══════════════════════════════════════════════════════════════════════
# 6. 메인 함수
# ═══════════════════════════════════════════════════════════════════════
def main():
    # EXAONE 4.0 연결
    logger.info("="*60)
    logger.info("🔷 EXAONE 4.0.1 서버 연결 중...")
    logger.info("="*60)

    client = openai.OpenAI(
        base_url="http://218.156.182.231:1234/v1",
        api_key="not-needed"
    )

    # 연결 테스트
    try:
        test_response = client.chat.completions.create(
            model="lgai-exaone_exaone-4.0.1-32b",
            messages=[{"role": "user", "content": "안녕하세요"}],
            max_tokens=10
        )
        logger.info("✅ EXAONE 4.0.1 연결 성공!")
        logger.info("="*60 + "\n")
    except Exception as e:
        logger.error(f"❌ 연결 실패: {e}")
        exit(1)

    # 설정값
    INPUT_FILE = "/mnt/c/Users/com/Desktop/프로젝트/5.18뉴스+댓글데이터_241201-250515/1. data/202508-202509/merged_output_news_cleaned001.csv"
    OUTPUT_FILE = "merged_output_news_cleaned017.csv"
    OUTPUT_DIR = "/mnt/c/Users/com/Desktop/프로젝트/5.18뉴스+댓글데이터_241201-250515/1. data/202508-202509/"
    CHECKPOINT_INTERVAL = 50

    # 1. 데이터 로드
    df = load_and_clean_data(INPUT_FILE)

    # 2. 전체 분석
    logger.info("="*60)
    logger.info("🎯 EXAONE 4.0.1 전체 뉴스 요약 시작")
    logger.info("="*60)
    analyzer = NewsSummaryAnalyzer(client, output_dir=OUTPUT_DIR)

    result_df = analyzer.process_dataframe(
        df,
        output_file=OUTPUT_FILE,
        checkpoint_interval=CHECKPOINT_INTERVAL
    )

    # 3. 완료
    logger.info("🎊 모든 작업 완료!")

if __name__ == "__main__":
    main()

In [ ]:
{
  "type": "object",
  "properties": {
    "summary": {
      "type": "string",
      "minLength": 10,
      "maxLength": 512,
      "description": "뉴스 기사의 요약 (최대 3문장 이하, 중립적 문어체)"
    },
    "keywords": {
      "type": "array",
      "items": {
        "type": "string",
        "minLength": 2,
        "maxLength": 50
      },
      "minItems": 3,
      "maxItems": 3,
      "description": "핵심 키워드 정확히 3개"
    }
  },
  "required": [
    "summary",
    "keywords"
  ],
  "additionalProperties": false
}